In [ ]:
import os
import requests
import pandas as pd
import datetime
import time

# --- 1. Configuration ---
SOURCE_PREFIX = "ELSST"
SOURCE_NAME = "European Language Social Science Thesaurus"
API_BASE = "https://thesauri.cessda.eu/rest/v1/elsst-6"
CONCEPT_BASE_URI = "https://elsst.cessda.eu/id/6/"

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))

# FILE NAMING
raw_ingest_file = os.path.join(raw_data_dir, "elsst_ingest_run.csv")
master_file = os.path.join(raw_data_dir, "metadata_ingest_undeduped.csv")
prefix_map_file = os.path.join(raw_data_dir, "prefix_map.csv")

# Verified Seed List
TOP_LEVEL_URIS = [
    "https://elsst.cessda.eu/id/6/28ce4564-fb7b-48a3-81ff-2e31dcfb508b", # BELIEFS
    "https://elsst.cessda.eu/id/6/0a7efa6d-2063-4ad2-b3cc-13e2306c2dc1", # CHURCH AND STATE
    "https://elsst.cessda.eu/id/6/b4067811-ede2-4338-a1bf-b751800ae394", # ESOTERIC PRACTICES
    "https://elsst.cessda.eu/id/6/a80484f0-51cd-4730-b544-1465510f7f8e", # ETHICS
    "https://elsst.cessda.eu/id/6/575d20f3-b0f9-4794-b776-1e120c557d00", # PHILOSOPHY
    "https://elsst.cessda.eu/id/6/6cf5a38e-d0cc-4b06-9917-659c66adadf1", # RELIGION
    "https://elsst.cessda.eu/id/6/ce795285-4d4d-496d-83be-d79d77c5baf9", # ANTI-SEMITISM
    "https://elsst.cessda.eu/id/6/444b37b0-9004-4975-8d02-d2651964c6ed", # RELIGIOUS LEADERS
    "https://elsst.cessda.eu/id/6/b8ae6bd1-45a1-43d4-a0b2-d753ed94d143", # FAITH SCHOOLS
    "https://elsst.cessda.eu/id/6/2bf02f5d-d189-4651-82ae-4cb9fd375b9a", # JEWS
    "https://elsst.cessda.eu/id/6/a63185a1-cc79-4c0c-abca-ed6cd0e40743", # RELIGION AND THEOLOGY EDUCATION
    "https://elsst.cessda.eu/id/6/49384517-5a79-4cae-bd92-3dbdb1d06431", # RELIGIOUS BUILDINGS
    "https://elsst.cessda.eu/id/6/21e90540-27f2-40c3-81e1-c045a06fb9ba", # RELIGIOUS DISCRIMINATION
    "https://elsst.cessda.eu/id/6/0519458c-418d-4a4f-810b-5c78cacf4c4a", # RELIGIOUS FREEDOM
    "https://elsst.cessda.eu/id/6/730df5ec-cb2e-4086-8f5d-f2c429fc4353", # RELIGIOUS INSTRUCTION
    "https://elsst.cessda.eu/id/6/583acf15-cabc-45ab-a4a8-0912814a01ea", # RELIGIOUS SOCIETIES
    "https://elsst.cessda.eu/id/6/17b4a01c-6277-46c3-88d5-dd47370d7dc2"  # SPIRITUAL HEALING
]

HEADERS = {'Accept': 'application/json'}
COLUMN_ORDER = ["Source_System", "Primary_Label", "CURIE", "Formal_Label", "Hierarchy_Path", "Synonyms", "Description", "Parent_IDs", "Concept_ID", "URI", "Status", "Extraction_Date"]

os.makedirs(raw_data_dir, exist_ok=True)

# --- 2. Prefix Map Update ---
new_row = {"Prefix": SOURCE_PREFIX, "Base_URI": CONCEPT_BASE_URI, "Source_Name": SOURCE_NAME}

if os.path.exists(prefix_map_file):
    df_prefixes = pd.read_csv(prefix_map_file)
    if SOURCE_PREFIX not in df_prefixes['Prefix'].values:
        df_prefixes = pd.concat([df_prefixes, pd.DataFrame([new_row])], ignore_index=True)
        df_prefixes.to_csv(prefix_map_file, index=False, encoding='utf-8-sig')
        print(f"Added {SOURCE_PREFIX} to prefix_map.csv")
else:
    df_prefixes = pd.DataFrame([new_row])
    df_prefixes.to_csv(prefix_map_file, index=False, encoding='utf-8-sig')
    print(f"Created prefix_map.csv and registered {SOURCE_PREFIX}.")

print(f"ELSST Ready.\nRaw: {raw_ingest_file}\nMaster: {master_file}")

ELSST Ready.
Raw: c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\elsst_ingest_run.csv
Master: c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\metadata_ingest_undeduped.csv


In [ ]:
processed_ids = set()
label_cache = {}

def get_english_value(data_item):
    """Extracts English prefLabel or definition from ELSST's multilingual arrays."""
    if isinstance(data_item, list):
        for entry in data_item:
            if isinstance(entry, dict) and entry.get('lang') == 'en':
                return entry.get('value', '')
    elif isinstance(data_item, dict):
        return data_item.get('value', '')
    return str(data_item) if data_item else ""

def get_parent_info(uri):
    """Looks 'up' the tree once to find the parent for a specific seed."""
    url = f"{API_BASE}/data?uri={uri}&lang=en"
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            graph = res.json().get('graph', [])
            node = next((n for n in graph if n.get('uri') == uri), {})
            broader = node.get('broader')
            if broader:
                b_uri = broader[0].get('uri') if isinstance(broader, list) else broader.get('uri')
                if b_uri:
                    b_label_url = f"{API_BASE}/label?uri={b_uri}&lang=en"
                    b_res = requests.get(b_label_url, headers=HEADERS, timeout=10)
                    b_label = b_res.json().get('prefLabel', 'Unknown Parent')
                    return b_uri.split('/')[-1], b_label
    except: pass
    return "", ""

def get_concept_details(uri):
    """Fetches full details for a concept node."""
    url = f"{API_BASE}/data?uri={uri}&lang=en"
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            graph = res.json().get('graph', [])
            node = next((n for n in graph if n.get('uri') == uri), {})
            label = get_english_value(node.get('prefLabel', ''))
            
            alts_raw = node.get('altLabel', [])
            if not isinstance(alts_raw, list): alts_raw = [alts_raw]
            synonyms = [a.get('value') for a in alts_raw if isinstance(a, dict) and a.get('lang') == 'en']
            
            defn = get_english_value(node.get('definition', ''))
            if not defn:
                defn = get_english_value(node.get('skos:scopeNote', ''))
                
            return {'label': label, 'synonyms': " | ".join(synonyms), 'description': defn}
    except: pass
    return None

def ingest_recursive(uri, path="", p_id=""):
    """Crawls down the 'narrower' (child) branches of the tree."""
    cid = uri.rstrip('/').split('/')[-1]
    if cid in processed_ids: return
    
    details = get_concept_details(uri)
    if not details or not details['label']: return
    
    current_label = details['label']
    current_path = f"{path} > {current_label}" if path else current_label
    
    display_text = f"Ingesting: {current_path[:90]}..."
    print(f"\r{display_text}{' ' * 60}", end="", flush=True)
    
    row = {
        'Source_System': SOURCE_NAME, 'Primary_Label': current_label,
        'CURIE': f"{SOURCE_PREFIX}:{cid}", 'Formal_Label': current_label,
        'Hierarchy_Path': current_path, 'Synonyms': details['synonyms'],
        'Description': details['description'], 'Parent_IDs': p_id,
        'Concept_ID': cid, 'URI': uri, 'Status': 'active',
        'Extraction_Date': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    pd.DataFrame([row])[COLUMN_ORDER].to_csv(raw_ingest_file, mode='a', index=False, 
                                          header=not os.path.exists(raw_ingest_file), encoding='utf-8-sig')
    processed_ids.add(cid)

    narrower_url = f"{API_BASE}/narrower?uri={uri}&lang=en"
    try:
        n_res = requests.get(narrower_url, headers=HEADERS, timeout=10)
        if n_res.status_code == 200:
            for child in n_res.json().get('narrower', []):
                child_uri = child.get('uri')
                if child_uri:
                    time.sleep(0.1)
                    ingest_recursive(child_uri, current_path, cid)
    except: pass

# --- Execution ---
if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)

print(f"Starting ELSST-6 Hierarchy-Aware Ingest...")

try:
    for uri in TOP_LEVEL_URIS:
        parent_id, parent_label = get_parent_info(uri)
        ingest_recursive(uri, path=parent_label, p_id=parent_id)
finally:
    print(f"\n\nIngest Complete! Data saved to: {os.path.basename(raw_ingest_file)}")

Starting ELSST-6 Hierarchy-Aware Harvest...
Harvesting: COMPLEMENTARY THERAPIES > SPIRITUAL HEALING...                                                                                                      

✅ Harvest Complete! Data saved to: elsst_ingest_run.csv


In [ ]:
if os.path.exists(raw_ingest_file):
    df_new = pd.read_csv(raw_ingest_file)
    
    if os.path.exists(master_file):
        df_master = pd.read_csv(master_file)
        existing_uris = set(df_master['URI'].dropna().unique())
        df_to_add = df_new[~df_new['URI'].isin(existing_uris)]
        
        if not df_to_add.empty:
            pd.concat([df_master, df_to_add], ignore_index=True).to_csv(master_file, index=False, encoding='utf-8-sig')
            print(f"Added {len(df_to_add)} records to master.")
        else:
            print("No new records found.")
    else:
        df_new.to_csv(master_file, index=False, encoding='utf-8-sig')
        print(f"Master file created.")
else:
    print(f"Error: {raw_ingest_file} not found. Please run the ingest cell first.")

✅ Added 111 records to master.
